In [0]:
from pyspark.sql.functions import col, to_timestamp, split, lower, when, count, countDistinct, sum

SILVER_CHECKPOINT_PATH = "/Volumes/workspace/bronze_cosmetics/checkpoints/silver"
BRONZE_TABLE = "workspace.bronze_cosmetics.cosmetics_events_bronze"
SILVER_TABLE = "workspace.silver_cosmetics.cosmetics_events_silver"


print("Đang định nghĩa logic Blacklist Bot và Spam Session...")
bronze_batch_df = spark.read.table(BRONZE_TABLE)

# 1.1 Định vị Bot User (Quy tắc: > 100 click/phiên)
bot_users_df = (bronze_batch_df
    .groupBy("user_id")
    .agg((count("event_time") / countDistinct("user_session")).alias("clicks_per_session"))
    .filter(col("clicks_per_session") > 100)
    .select("user_id")
)

# 1.2 Định vị Phiên ảo/Spam API (Quy tắc: > 50 add-to-cart nhưng 0 view, 0 purchase)
spam_sessions_df = (bronze_batch_df
    .groupBy("user_session")
    .agg(
        sum(when(col("event_type") == "cart", 1).otherwise(0)).alias("count_carts"),
        sum(when(col("event_type") == "view", 1).otherwise(0)).alias("count_views"),
        sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("count_purchases")
    )
    .filter((col("count_carts") > 50) & (col("count_views") == 0) & (col("count_purchases") == 0))
    .select("user_session")
)

print("Logic Blacklist đã sẵn sàng. Khởi động luồng Streaming...")


print(f"Đang đọc luồng dữ liệu từ bảng Bronze: {BRONZE_TABLE}...")
bronze_stream_df = spark.readStream.table(BRONZE_TABLE)

silver_df = (bronze_stream_df
    
    # LỌC BOT VÀ SPAM BẰNG LEFT ANTI JOIN 
    .join(bot_users_df, on="user_id", how="left_anti")
    .join(spam_sessions_df, on="user_session", how="left_anti")
    
    # CHUẨN HÓA KIỂU DỮ LIỆU 
    .withColumn("event_time", to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss 'UTC'"))
    .withColumn("price", col("price").cast("float"))
    
    .withColumn("brand", when((col("brand").isNull()) | (col("brand") == ""), "unbranded")
                         .otherwise(lower(col("brand"))))
    
    .withColumn("category_code", when((col("category_code").isNull()) | (col("category_code") == ""), "accessories.other")
                                      .otherwise(col("category_code")))
    
    .withColumn("main_category", split(col("category_code"), "\\.")[0])
    .withColumn("sub_category", split(col("category_code"), "\\.")[1])
    
    .select(
        "event_time", 
        "event_type", 
        "product_id", 
        "category_id",     
        "main_category",   
        "sub_category",    
        "brand",          
        "price", 
        "user_id", 
        "user_session"
    )

    .filter(
        col("user_id").isNotNull() & 
        col("product_id").isNotNull() & 
        (col("price") > 0)
    )
    
    .dropDuplicates([
        "event_time", "event_type", "product_id", 
        "category_id", "brand", "price", "user_id", "user_session"
    ])
)

print(f"Đang thực thi quá trình làm sạch và ghi vào bảng Silver: {SILVER_TABLE}...")

query_silver = (silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", SILVER_CHECKPOINT_PATH)
    .trigger(availableNow=True) 
    .table(SILVER_TABLE)
)

# Chờ tiến trình hoàn tất
query_silver.awaitTermination()

print("Dữ liệu đã được nạp vào tầng Silver thành công")

Đang định nghĩa logic Blacklist Bot và Spam Session...
Logic Blacklist đã sẵn sàng. Khởi động luồng Streaming...
Đang đọc luồng dữ liệu từ bảng Bronze: workspace.bronze_cosmetics.cosmetics_events_bronze...
Đang thực thi quá trình làm sạch và ghi vào bảng Silver: workspace.silver_cosmetics.cosmetics_events_silver...


26/06/12 05:30:13 Spark Server has not sent updates for Streaming Query 722180dd-4fb5-44bf-9472-f1e1338f543c in 60 seconds, but the query is still active. Marking query as in-progress. Spark Session ID is 0a332085-6988-4c47-9ffb-60d38248fece. This is typically not a problem.


Dữ liệu đã được nạp vào tầng Silver thành công


In [0]:
# 1. Đọc lại 2 bảng dưới dạng Batch để đếm số liệu tĩnh
bronze_batch = spark.read.table(BRONZE_TABLE)
silver_batch = spark.read.table(SILVER_TABLE)

# 2. Thống kê bảng Bronze 
bronze_monthly_count = (bronze_batch
    .withColumn("parsed_time", to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss 'UTC'"))
    .withColumn("month_year", date_format(col("parsed_time"), "yyyy-MM"))
    .groupBy("month_year")
    .count()
    .withColumnRenamed("count", "bronze_total_rows")
)

# 3. Thống kê bảng Silver 
silver_monthly_count = (silver_batch
    .withColumn("month_year", date_format(col("event_time"), "yyyy-MM"))
    .groupBy("month_year")
    .count()
    .withColumnRenamed("count", "silver_total_rows")
)

# 4. Gộp 2 bảng thống kê để so sánh
reconciliation_df = (bronze_monthly_count
    .join(silver_monthly_count, on="month_year", how="outer")
    .orderBy("month_year")
)

# 5. Tính toán chênh lệch 
reconciliation_df = reconciliation_df.withColumn(
    "rows_filtered_out",
    col("bronze_total_rows") - col("silver_total_rows")
)

# 6. Hiển thị bảng báo cáo
reconciliation_df.show(truncate=False)

print("""
Ghi chú:
- 'bronze_total_rows': Tổng số sự kiện thô đẩy lên từ Kafka.
- 'silver_total_rows': Tổng số sự kiện sạch được giữ lại để đưa lên Gold.
- 'rows_filtered_out': Đây là lượng rác (Null User/Product) và các dòng bị trùng lặp kỹ thuật (Kafka gửi đúp) mà pipeline của bạn đã dọn dẹp thành công.
""")

+----------+-----------------+-----------------+-----------------+
|month_year|bronze_total_rows|silver_total_rows|rows_filtered_out|
+----------+-----------------+-----------------+-----------------+
|NULL      |68               |NULL             |NULL             |
|2019-10   |6499536          |3149531          |3350005          |
|2019-11   |9206758          |4171262          |5035496          |
|2019-12   |8659529          |3348887          |5310642          |
+----------+-----------------+-----------------+-----------------+


Ghi chú:
- 'bronze_total_rows': Tổng số sự kiện thô đẩy lên từ Kafka.
- 'silver_total_rows': Tổng số sự kiện sạch được giữ lại để đưa lên Gold.
- 'rows_filtered_out': Đây là lượng rác (Null User/Product) và các dòng bị trùng lặp kỹ thuật (Kafka gửi đúp) mà pipeline của bạn đã dọn dẹp thành công.

